# Decision fusion, final system

Part of the code for *Danger Detection through Audio Captioning and Classification Fusion* (Electronics, MDPI). Paper section: 3.4, 4.4 (Tables 10–12).

Combines the CNN and DistilBERT predictions on UrbanSound8K under the twelve fusion rules. The final system is LV-18 (weighted CE) + DistilBERT (alignment-stratified, weighted CE) with the custom_uncertain rule and activation zone 0.17–0.60, derived on the internal test subset (see derive_fusion_thresholds.py).

**Environment.** Written for Google Colab (CPU is enough); see `environment/`. Paths point to a Google Drive layout. Edit the path cell to your own copy of the datasets. Random seed: 42.

**Inputs.** CNN and DistilBERT prediction files on UrbanSound8K (8419 clips).

**Outputs.** Fusion metrics for every rule and model pair.

**Note.** This notebook compares all rules and model pairs; its custom_uncertain zone for LV-18 is the exploratory 0.17–0.73. The final results reported in the paper (zone 0.17–0.60, Tables 9, 10 and 12, bootstrap CIs) are produced by `final_fusion_evaluation.py`.


# Weighted CE Selected CNN + Random DistilBERT Fusion Rules

This notebook evaluates fusion rules for the selected Weighted CE CNNs only:

- `DangerDetective_1DWAVE_LV-18_s1_p8__selected_by_val_auc_roc`
- `DangerDetective_1DWAVE_LV-24_s2_p4__selected_by_val_auc_roc`

It uses only the random DistilBERT training schemes:

- `low_threshold__random__focal_loss`
- `low_threshold__random__standard_ce`
- `low_threshold__random__weighted_ce`

CNN-specific threshold windows are used for `probabilistic_late`, `custom_uncertain_handoff`, and `cascade_near_boundary_rescue`.


## 1. Mount Drive and Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_rows', 80)


## 2. Editable Paths and Model Lists

Edit `BASE_DIR` and `OUTPUT_DIR` for your Colab/Drive location. The folder names and CSV names are set to match your local structure.


In [ ]:
# EDIT OUTPUT_DIR IF YOU WANT A DIFFERENT OUTPUT LOCATION
CNN_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/Weighted_CE_CNN/UrbanSound_unknown_test')
DISTILBERT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/DistilBERT/training/low_threshold_experiments/external_eval_urbansound')
OUTPUT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/Weighted_CE_CNN/Fusion_rules/FINAL/SELECTED_ONLY')

CNN_PREDICTION_CSV = 'urbansound_unknown_test_predictions_detailed.csv'
DISTILBERT_PREDICTION_CSV = 'urbansound8k_predictions.csv'

PLOT_DIR = OUTPUT_DIR / 'plots'
PAIR_DIR = OUTPUT_DIR / 'pair_matrices'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
PAIR_DIR.mkdir(parents=True, exist_ok=True)

CNN_DIRS = [
    'DangerDetective_1DWAVE_LV-18_s1_p8__selected_by_val_auc_roc',
    'DangerDetective_1DWAVE_LV-24_s2_p4__selected_by_val_auc_roc',
]
DISTILBERT_DIRS = [
    'low_threshold__random__focal_loss',
    'low_threshold__random__standard_ce',
    'low_threshold__random__weighted_ce',
]
print('CNN_ROOT =', CNN_ROOT)
print('DISTILBERT_ROOT =', DISTILBERT_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('CNN models:', len(CNN_DIRS))
print('DistilBERT models:', len(DISTILBERT_DIRS))

## 3. Fusion Rule Settings

In [ ]:
CNN_THRESHOLD = 0.50
FUSION_THRESHOLD = 0.50
MODEL_SPECIFIC_THRESHOLDS = {
    'LV-18': {
        'probabilistic_late_low': 0.17,
        'probabilistic_late_high': 0.50,
        'uncertain_low': 0.17,
        'uncertain_high': 0.73,
    },
    'LV-24': {
        'probabilistic_late_low': 0.34,
        'probabilistic_late_high': 0.50,
        'uncertain_low': 0.34,
        'uncertain_high': 0.65,
    },
}

CNN_WEIGHT = 0.50
DIST_WEIGHT = 0.50
UNKNOWN_PRED = -1
DS_UNCERTAINTY_CNN = 0.10
DS_UNCERTAINTY_DISTILBERT = 0.05

RULE_DESCRIPTIONS = pd.DataFrame([
    {'rule': 'cnn_only', 'meaning': 'Use CNN probability only.'},
    {'rule': 'distilbert_only', 'meaning': 'Use DistilBERT probability only.'},
    {'rule': 'override', 'meaning': 'If DistilBERT predicts danger, use DistilBERT probability; otherwise keep CNN probability.'},
    {'rule': 'weighted_average', 'meaning': 'Average CNN and DistilBERT probabilities with equal weights.'},
    {'rule': 'cascade_correction', 'meaning': 'If CNN probability is in the model-specific low-confidence window and DistilBERT predicts danger, use DistilBERT probability; otherwise keep CNN probability. LV18: 0.17 <= CNN <= 0.50. LV24: 0.34 <= CNN <= 0.50.'},
    {'rule': 'probabilistic_late', 'meaning': 'If CNN probability is in the model-specific low-confidence window, use weighted average probability; otherwise keep CNN probability. LV18: 0.17 <= CNN <= 0.50. LV24: 0.34 <= CNN <= 0.50.'},
    {'rule': 'custom_uncertain_handoff', 'meaning': 'If CNN probability is in the model-specific uncertainty window, use DistilBERT probability; otherwise keep CNN probability. LV18: 0.17 <= CNN <= 0.73. LV24: 0.34 <= CNN <= 0.65.'},
    {'rule': 'cascade_near_boundary_rescue', 'meaning': 'If CNN probability is in the model-specific uncertainty window and DistilBERT predicts danger, use DistilBERT probability; otherwise keep CNN probability. LV18: 0.17 <= CNN <= 0.73. LV24: 0.34 <= CNN <= 0.65.'},
    {'rule': 'ds_fusion', 'meaning': 'Dempster-Shafer late fusion using uncertainty masses 0.10 for CNN and 0.05 for DistilBERT.'},
    {'rule': 'ds_fusion_no_theta', 'meaning': 'Dempster-Shafer-style fusion over danger and safe only, with no uncertainty/theta mass.'},
    {'rule': 'or_rule', 'meaning': 'If either CNN or DistilBERT predicts danger, output danger; if both predict no danger, output no danger.'},
    {'rule': 'and_rule', 'meaning': 'If both models agree, output their shared class; if they disagree, output unknown.'},
])

RULE_DESCRIPTIONS


## 4. Helper Functions

In [ ]:
def slug(text: str) -> str:
    text = re.sub(r'[^A-Za-z0-9]+', '_', str(text)).strip('_')
    return re.sub(r'_+', '_', text).lower()


def model_short_name(name: str) -> str:
    return (
        str(name)
        .replace('DangerDetective_', '')
        .replace('low_threshold__', '')
        .replace('__selected_by_', '__sel_')
        .replace('__best_epoch', '__best')
        .replace('alignment_stratified', 'align')
    )


def normalize_label(values: pd.Series) -> pd.Series:
    raw = values.astype(str).str.strip().str.lower()
    return raw.replace({
        '0': 'no_danger_risk',
        '1': 'danger_risk',
        'no danger risk': 'no_danger_risk',
        'danger risk': 'danger_risk',
        'not_danger_related': 'no_danger_risk',
        'danger_related': 'danger_risk',
    })


def first_existing_column(df: pd.DataFrame, candidates: list[str], required: bool = True) -> str | None:
    for col in candidates:
        if col in df.columns:
            return col
    if required:
        raise KeyError(f'None of these columns exist: {candidates}. Available columns: {list(df.columns)}')
    return None


def basename_series(values: pd.Series) -> pd.Series:
    return values.astype(str).map(lambda x: Path(x).name)


def compute_metrics(y_true, prob, pred) -> dict:
    y_true_all = pd.to_numeric(pd.Series(y_true), errors='coerce').astype(int)
    prob_all = pd.to_numeric(pd.Series(prob), errors='coerce')
    pred_all = pd.to_numeric(pd.Series(pred), errors='coerce').astype(int)

    known_mask = pred_all.isin([0, 1])
    y_true = y_true_all[known_mask]
    prob = prob_all[known_mask]
    pred = pred_all[known_mask]

    result = {
        'total_rows': int(len(pred_all)),
        'evaluated_rows': int(known_mask.sum()),
        'unknown_rows': int((~known_mask).sum()),
        'coverage_rate': float(known_mask.mean()) if len(known_mask) else np.nan,
    }

    if len(pred) == 0:
        result.update({
            'accuracy': np.nan,
            'macro_f1': np.nan,
            'precision_no_danger': np.nan,
            'recall_no_danger': np.nan,
            'f1_no_danger': np.nan,
            'support_no_danger': 0,
            'precision_danger': np.nan,
            'recall_danger': np.nan,
            'f1_danger': np.nan,
            'support_danger': 0,
            'tn': 0,
            'fp': 0,
            'fn': 0,
            'tp': 0,
            'false_positive_rate': np.nan,
            'false_negative_rate': np.nan,
            'roc_auc': np.nan,
            'average_precision': np.nan,
        })
        return result

    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, pred, labels=[0, 1], zero_division=0
    )

    result.update({
        'accuracy': accuracy_score(y_true, pred),
        'macro_f1': float((f1[0] + f1[1]) / 2),
        'precision_no_danger': precision[0],
        'recall_no_danger': recall[0],
        'f1_no_danger': f1[0],
        'support_no_danger': int(support[0]),
        'precision_danger': precision[1],
        'recall_danger': recall[1],
        'f1_danger': f1[1],
        'support_danger': int(support[1]),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
        'false_positive_rate': int(cm[0, 1]) / max(int(cm[0, 0] + cm[0, 1]), 1),
        'false_negative_rate': int(cm[1, 0]) / max(int(cm[1, 0] + cm[1, 1]), 1),
    })
    if y_true.nunique() == 2 and prob.notna().all():
        result['roc_auc'] = roc_auc_score(y_true, prob)
        result['average_precision'] = average_precision_score(y_true, prob)
    else:
        result['roc_auc'] = np.nan
        result['average_precision'] = np.nan
    return result


## 5. Load and Standardize Predictions

In [ ]:
def load_cnn_model(model_dir_name: str) -> pd.DataFrame:
    path = CNN_ROOT / model_dir_name / CNN_PREDICTION_CSV
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)

    filename_col = first_existing_column(df, ['filename', 'slice_file_name', 'source_filename', 'filepath', 'path'])
    class_col = first_existing_column(df, ['class', 'category', 'original_category'], required=False)
    caption_col = first_existing_column(df, ['caption'], required=False)
    y_col = first_existing_column(df, ['y_true', 'label_int', 'danger_label'])
    label_col = first_existing_column(df, ['semantic_label_final', 'risk_label', 'label'], required=False)
    prob_col = first_existing_column(df, ['prob_danger_risk', 'danger_prob', 'prob_danger', 'probability_danger'])
    pred_col = first_existing_column(df, ['y_pred', 'prediction', 'predicted_label'], required=False)

    out = pd.DataFrame({
        'filename': basename_series(df[filename_col]),
        'caption': df[caption_col].astype(str) if caption_col else '',
        'class': df[class_col].astype(str) if class_col else '',
        'y_true': pd.to_numeric(df[y_col], errors='coerce').astype(int),
        'reference_label': normalize_label(df[label_col]) if label_col else pd.to_numeric(df[y_col], errors='coerce').astype(int).map({0: 'no_danger_risk', 1: 'danger_risk'}),
        'model_family': 'CNN',
        'model_name': model_dir_name,
        'model_short': model_short_name(model_dir_name),
        'prob_danger': pd.to_numeric(df[prob_col], errors='coerce'),
    })

    if pred_col:
        out['prediction'] = pd.to_numeric(df[pred_col], errors='coerce').fillna(out['prob_danger'] >= CNN_THRESHOLD).astype(int)
    else:
        out['prediction'] = (out['prob_danger'] >= CNN_THRESHOLD).astype(int)
    out['prediction_label'] = out['prediction'].map({0: 'no_danger_risk', 1: 'danger_risk'})
    return out


def load_distilbert_model(model_dir_name: str) -> pd.DataFrame:
    path = DISTILBERT_ROOT / model_dir_name / DISTILBERT_PREDICTION_CSV
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)

    filename_col = first_existing_column(df, ['filename', 'slice_file_name', 'source_filename', 'filepath', 'path'])
    class_col = first_existing_column(df, ['class', 'category', 'original_class_for_eval', 'original_category'], required=False)
    caption_col = first_existing_column(df, ['caption'], required=False)
    y_col = first_existing_column(df, ['y_true', 'label_int', 'danger_label'])
    label_col = first_existing_column(df, ['semantic_label_final', 'risk_label', 'label'], required=False)
    prob_col = first_existing_column(df, ['distilbert_danger_prob', 'prob_danger_risk', 'danger_prob', 'prob_danger'])
    pred_col = first_existing_column(df, ['distilbert_prediction', 'prediction', 'y_pred'], required=False)

    out = pd.DataFrame({
        'filename': basename_series(df[filename_col]),
        'caption': df[caption_col].astype(str) if caption_col else '',
        'class': df[class_col].astype(str) if class_col else '',
        'y_true': pd.to_numeric(df[y_col], errors='coerce').astype(int),
        'reference_label': normalize_label(df[label_col]) if label_col else pd.to_numeric(df[y_col], errors='coerce').astype(int).map({0: 'no_danger_risk', 1: 'danger_risk'}),
        'model_family': 'DistilBERT',
        'model_name': model_dir_name,
        'model_short': model_short_name(model_dir_name),
        'prob_danger': pd.to_numeric(df[prob_col], errors='coerce'),
    })

    if pred_col:
        out['prediction'] = pd.to_numeric(df[pred_col], errors='coerce').fillna(out['prob_danger'] >= FUSION_THRESHOLD).astype(int)
    else:
        out['prediction'] = (out['prob_danger'] >= FUSION_THRESHOLD).astype(int)
    out['prediction_label'] = out['prediction'].map({0: 'no_danger_risk', 1: 'danger_risk'})
    return out


cnn_df = pd.concat([load_cnn_model(name) for name in CNN_DIRS], ignore_index=True)
dist_df = pd.concat([load_distilbert_model(name) for name in DISTILBERT_DIRS], ignore_index=True)

print('CNN models:', cnn_df['model_name'].nunique(), 'rows:', len(cnn_df))
print('DistilBERT models:', dist_df['model_name'].nunique(), 'rows:', len(dist_df))
print('Unique files in CNN:', cnn_df['filename'].nunique())
print('Unique files in DistilBERT:', dist_df['filename'].nunique())

cnn_df.head()


## 6. Fusion Rule Implementation

In [ ]:
RULE_PROB_COLS = [
    'rule_cnn_only_prob',
    'rule_distilbert_only_prob',
    'rule_override_prob',
    'rule_weighted_average_prob',
    'rule_cascade_correction_prob',
    'rule_probabilistic_late_prob',
    'rule_custom_uncertain_handoff_prob',
    'rule_cascade_near_boundary_rescue_prob',
    'rule_ds_fusion_prob',
    'rule_ds_fusion_no_theta_prob',
    'rule_or_rule_prob',
    'rule_and_rule_prob',
]


def thresholds_for_cnn(cnn_name: str) -> dict:
    cnn_name = str(cnn_name)
    if 'LV-18' in cnn_name:
        return MODEL_SPECIFIC_THRESHOLDS['LV-18']
    if 'LV-24' in cnn_name:
        return MODEL_SPECIFIC_THRESHOLDS['LV-24']
    raise ValueError(f'No model-specific fusion thresholds configured for CNN model: {cnn_name}')


def fusion_probabilities(pair_df: pd.DataFrame, cnn_name: str) -> pd.DataFrame:
    out = pair_df.copy()
    cnn = out['cnn_prob_danger']
    dist = out['dist_prob_danger']
    cnn_pred = out['cnn_pred'].astype(int)
    dist_pred = out['dist_pred'].astype(int)
    weighted = ((CNN_WEIGHT * cnn) + (DIST_WEIGHT * dist)) / (CNN_WEIGHT + DIST_WEIGHT)

    thresholds = thresholds_for_cnn(cnn_name)
    cascade = (
        (cnn >= thresholds['probabilistic_late_low'])
        & (cnn <= thresholds['probabilistic_late_high'])
    )
    probabilistic_late_window = cascade
    uncertain = (
        (cnn >= thresholds['uncertain_low'])
        & (cnn <= thresholds['uncertain_high'])
    )
    near_boundary = uncertain
    models_agree = cnn_pred.eq(dist_pred)

    out['rule_cnn_only_prob'] = cnn
    out['rule_distilbert_only_prob'] = dist
    out['rule_override_prob'] = np.where(dist_pred == 1, dist, cnn)
    out['rule_weighted_average_prob'] = weighted
    out['rule_cascade_correction_prob'] = np.where(cascade & (dist_pred == 1), dist, cnn)
    out['rule_probabilistic_late_prob'] = np.where(probabilistic_late_window, weighted, cnn)
    out['rule_custom_uncertain_handoff_prob'] = np.where(uncertain, dist, cnn)
    out['rule_cascade_near_boundary_rescue_prob'] = np.where(near_boundary & (dist_pred == 1), dist, cnn)

    m1_danger = cnn * (1 - DS_UNCERTAINTY_CNN)
    m1_safe = (1 - cnn) * (1 - DS_UNCERTAINTY_CNN)
    m1_theta = DS_UNCERTAINTY_CNN
    m2_danger = dist * (1 - DS_UNCERTAINTY_DISTILBERT)
    m2_safe = (1 - dist) * (1 - DS_UNCERTAINTY_DISTILBERT)
    m2_theta = DS_UNCERTAINTY_DISTILBERT
    conflict = (m1_danger * m2_safe) + (m1_safe * m2_danger)
    k = 1 - conflict

    fused_danger = (
        (m1_danger * m2_danger) + (m1_danger * m2_theta) + (m1_theta * m2_danger)
    ) / k
    fused_safe = (
        (m1_safe * m2_safe) + (m1_safe * m2_theta) + (m1_theta * m2_safe)
    ) / k

    out['rule_ds_fusion_prob'] = fused_danger
    out['rule_ds_fusion_pred'] = np.where(fused_danger > fused_safe, 1, 0).astype(int)

    no_theta_conflict = (cnn * (1 - dist)) + ((1 - cnn) * dist)
    no_theta_k = np.maximum(1 - no_theta_conflict, 1e-12)
    no_theta_danger = (cnn * dist) / no_theta_k
    no_theta_safe = ((1 - cnn) * (1 - dist)) / no_theta_k
    out['rule_ds_fusion_no_theta_prob'] = no_theta_danger
    out['rule_ds_fusion_no_theta_pred'] = np.where(
        no_theta_danger > no_theta_safe, 1, 0
    ).astype(int)

    out['rule_or_rule_pred'] = np.where((cnn_pred == 1) | (dist_pred == 1), 1, 0).astype(int)
    out['rule_or_rule_prob'] = np.maximum(cnn, dist)

    out['rule_and_rule_pred'] = np.where(models_agree, cnn_pred, UNKNOWN_PRED).astype(int)
    out['rule_and_rule_prob'] = np.where(
        models_agree,
        np.where(cnn_pred == 1, np.minimum(cnn, dist), np.maximum(cnn, dist)),
        np.nan,
    )
    out['rule_and_rule_label'] = pd.Series(out['rule_and_rule_pred']).map({
        0: 'no_danger_risk',
        1: 'danger_risk',
        UNKNOWN_PRED: 'unknown',
    })

    out['cascade_triggered'] = cascade
    out['probabilistic_late_triggered'] = probabilistic_late_window
    out['custom_uncertain_triggered'] = uncertain
    out['near_boundary_rescue_triggered'] = near_boundary
    out['near_boundary_rescue_used_distilbert'] = near_boundary & (dist_pred == 1)
    out['or_rule_triggered'] = cnn_pred.ne(dist_pred)
    out['and_rule_unknown'] = out['rule_and_rule_pred'].eq(UNKNOWN_PRED)

    for col in RULE_PROB_COLS:
        pred_col = col.replace('_prob', '_pred')
        if pred_col not in out.columns:
            out[pred_col] = (out[col] >= FUSION_THRESHOLD).astype(int)
    return out


def build_pair_matrix(cnn_model: pd.DataFrame, dist_model: pd.DataFrame) -> pd.DataFrame:
    cnn_cols = ['filename', 'caption', 'class', 'y_true', 'reference_label', 'prob_danger', 'prediction']
    dist_cols = ['filename', 'prob_danger', 'prediction']
    merged = cnn_model[cnn_cols].merge(
        dist_model[dist_cols],
        on='filename',
        how='inner',
        suffixes=('_cnn', '_dist'),
    )
    merged = merged.rename(columns={
        'prob_danger_cnn': 'cnn_prob_danger',
        'prob_danger_dist': 'dist_prob_danger',
        'prediction_cnn': 'cnn_pred',
        'prediction_dist': 'dist_pred',
    })
    cnn_name = str(cnn_model['model_name'].iloc[0]) if 'model_name' in cnn_model.columns else ''
    return fusion_probabilities(merged, cnn_name)


## 7. Evaluate Every CNN x DistilBERT Pair

In [ ]:
metric_rows = []
diagnostic_rows = []
matrix_frames = []

for cnn_name, cnn_model in cnn_df.groupby('model_name', sort=False):
    for dist_name, dist_model in dist_df.groupby('model_name', sort=False):
        pair = build_pair_matrix(cnn_model, dist_model)

        cnn_short = model_short_name(cnn_name)
        dist_short = model_short_name(dist_name)
        pair_name = f'{cnn_short}__X__{dist_short}'
        pair.insert(0, 'cnn_model', cnn_name)
        pair.insert(1, 'distilbert_model', dist_name)
        pair.insert(2, 'pair_name', pair_name)

        matrix_cols = [
            'pair_name', 'cnn_model', 'distilbert_model', 'filename', 'class', 'caption',
            'y_true', 'reference_label', 'cnn_prob_danger', 'dist_prob_danger',
            *RULE_PROB_COLS,
            'cnn_pred', 'dist_pred',
            *[c.replace('_prob', '_pred') for c in RULE_PROB_COLS],
            'rule_and_rule_label',
            'cascade_triggered', 'probabilistic_late_triggered', 'custom_uncertain_triggered',
            'near_boundary_rescue_triggered', 'near_boundary_rescue_used_distilbert',
            'or_rule_triggered', 'and_rule_unknown',
        ]
        pair[matrix_cols].to_csv(PAIR_DIR / f'{slug(pair_name)}_fusion_matrix.csv', index=False)
        matrix_frames.append(pair[matrix_cols])

        for rule_col in RULE_PROB_COLS:
            rule_name = rule_col.replace('rule_', '').replace('_prob', '')
            pred_col = rule_col.replace('_prob', '_pred')
            metrics = compute_metrics(pair['y_true'], pair[rule_col], pair[pred_col])

            cnn_wrong = pair['cnn_pred'].ne(pair['y_true'])
            fusion_unknown = pair[pred_col].eq(UNKNOWN_PRED)
            fusion_correct = pair[pred_col].eq(pair['y_true'])

            metrics.update({
                'cnn_model': cnn_name,
                'distilbert_model': dist_name,
                'pair_name': pair_name,
                'rule_name': rule_name,
                'n_rows': len(pair),
                'cnn_misclassified_rows': int(cnn_wrong.sum()),
                'cnn_misclassified_fixed_by_fusion_rows': int((cnn_wrong & fusion_correct).sum()),
                'cnn_misclassified_not_fixed_by_fusion_rows': int((cnn_wrong & ~fusion_correct).sum()),
                'cnn_misclassified_unknown_by_fusion_rows': int((cnn_wrong & fusion_unknown).sum()),
                'cnn_correct_hurt_by_fusion_rows': int((~cnn_wrong & ~fusion_correct & ~fusion_unknown).sum()),
            })

            if rule_name == 'cascade_correction':
                trigger = pair['cascade_triggered']
                used = trigger & (pair['dist_pred'] == 1)
            elif rule_name == 'probabilistic_late':
                trigger = pair['probabilistic_late_triggered']
                used = trigger
            elif rule_name == 'custom_uncertain_handoff':
                trigger = pair['custom_uncertain_triggered']
                used = trigger
            elif rule_name == 'cascade_near_boundary_rescue':
                trigger = pair['near_boundary_rescue_triggered']
                used = pair['near_boundary_rescue_used_distilbert']
            elif rule_name in ['ds_fusion', 'ds_fusion_no_theta']:
                trigger = pd.Series(True, index=pair.index)
                used = trigger
            elif rule_name == 'or_rule':
                trigger = pair['or_rule_triggered']
                used = trigger
            elif rule_name == 'and_rule':
                trigger = pair['and_rule_unknown']
                used = ~pair['and_rule_unknown']
            else:
                trigger = pd.Series(False, index=pair.index)
                used = pd.Series(False, index=pair.index)

            metrics['triggered_rows'] = int(trigger.sum())
            metrics['used_distilbert_rows'] = int(used.sum())
            metric_rows.append(metrics)

            if trigger.any():
                triggered = pair[trigger].copy()
                cnn_correct = triggered['cnn_pred'].eq(triggered['y_true'])
                dist_correct = triggered['dist_pred'].eq(triggered['y_true'])
                triggered_fusion_unknown = triggered[pred_col].eq(UNKNOWN_PRED)
                triggered_fusion_correct = triggered[pred_col].eq(triggered['y_true'])
                diagnostic_rows.append({
                    'cnn_model': cnn_name,
                    'distilbert_model': dist_name,
                    'pair_name': pair_name,
                    'rule_name': rule_name,
                    'triggered_rows': int(trigger.sum()),
                    'used_distilbert_rows': int(used.sum()),
                    'unknown_rows_triggered': int(triggered_fusion_unknown.sum()),
                    'triggered_rate': float(trigger.mean()),
                    'used_distilbert_rate': float(used.mean()),
                    'cnn_accuracy_triggered': float(cnn_correct.mean()),
                    'distilbert_accuracy_triggered': float(dist_correct.mean()),
                    'fusion_accuracy_triggered': float(triggered_fusion_correct.mean()),
                    'fusion_helped_count': int((~cnn_correct & triggered_fusion_correct).sum()),
                    'fusion_hurt_count': int((cnn_correct & ~triggered_fusion_correct & ~triggered_fusion_unknown).sum()),
                    'fusion_unknown_on_cnn_wrong_count': int((~cnn_correct & triggered_fusion_unknown).sum()),
                    'fusion_net_help': int((~cnn_correct & triggered_fusion_correct).sum() - (cnn_correct & ~triggered_fusion_correct & ~triggered_fusion_unknown).sum()),
                    'changed_prediction_count': int(triggered['cnn_pred'].ne(triggered[pred_col]).sum()),
                })

all_metrics = pd.DataFrame(metric_rows).sort_values(
    ['macro_f1', 'accuracy', 'recall_danger'], ascending=False
)
all_diagnostics = pd.DataFrame(diagnostic_rows).sort_values(
    ['fusion_net_help', 'fusion_accuracy_triggered'], ascending=False
) if diagnostic_rows else pd.DataFrame()
all_pair_matrix_long = pd.concat(matrix_frames, ignore_index=True)

all_metrics.to_csv(OUTPUT_DIR / 'fusion_metrics_all_pairs_all_rules.csv', index=False)
all_diagnostics.to_csv(OUTPUT_DIR / 'fusion_rule_diagnostics.csv', index=False)
all_pair_matrix_long.to_csv(OUTPUT_DIR / 'fusion_probability_matrix_long.csv', index=False)

print('Pair matrices saved to:', PAIR_DIR)
print('Metrics saved to:', OUTPUT_DIR / 'fusion_metrics_all_pairs_all_rules.csv')
print('Diagnostics saved to:', OUTPUT_DIR / 'fusion_rule_diagnostics.csv')
print('Long matrix saved to:', OUTPUT_DIR / 'fusion_probability_matrix_long.csv')

all_metrics.head(30)


## 8. Best Rule Summaries

In [ ]:
best_rule_per_pair = (
    all_metrics
    .sort_values(['pair_name', 'macro_f1', 'accuracy', 'recall_danger'], ascending=[True, False, False, False])
    .groupby('pair_name', as_index=False)
    .head(1)
    .sort_values(['macro_f1', 'accuracy', 'recall_danger'], ascending=False)
)

best_pair_per_rule = (
    all_metrics
    .sort_values(['rule_name', 'macro_f1', 'accuracy', 'recall_danger'], ascending=[True, False, False, False])
    .groupby('rule_name', as_index=False)
    .head(1)
    .sort_values(['macro_f1', 'accuracy', 'recall_danger'], ascending=False)
)

rule_summary = (
    all_metrics
    .groupby('rule_name')
    .agg(
        mean_macro_f1=('macro_f1', 'mean'),
        max_macro_f1=('macro_f1', 'max'),
        mean_accuracy=('accuracy', 'mean'),
        mean_recall_danger=('recall_danger', 'mean'),
        mean_f1_danger=('f1_danger', 'mean'),
        mean_false_positive_rate=('false_positive_rate', 'mean'),
        mean_false_negative_rate=('false_negative_rate', 'mean'),
        mean_unknown_rows=('unknown_rows', 'mean'),
        mean_coverage_rate=('coverage_rate', 'mean'),
        mean_triggered_rows=('triggered_rows', 'mean'),
        mean_used_distilbert_rows=('used_distilbert_rows', 'mean'),
        mean_cnn_misclassified_fixed_by_fusion_rows=('cnn_misclassified_fixed_by_fusion_rows', 'mean'),
        mean_cnn_correct_hurt_by_fusion_rows=('cnn_correct_hurt_by_fusion_rows', 'mean'),
    )
    .reset_index()
    .sort_values(['max_macro_f1', 'mean_macro_f1'], ascending=False)
)

best_rule_per_pair.to_csv(OUTPUT_DIR / 'best_rule_per_pair.csv', index=False)
best_pair_per_rule.to_csv(OUTPUT_DIR / 'best_pair_per_rule.csv', index=False)
rule_summary.to_csv(OUTPUT_DIR / 'rule_summary.csv', index=False)

print('Best rule per pair saved to:', OUTPUT_DIR / 'best_rule_per_pair.csv')
print('Best pair per rule saved to:', OUTPUT_DIR / 'best_pair_per_rule.csv')
print('Rule summary saved to:', OUTPUT_DIR / 'rule_summary.csv')

rule_summary


In [ ]:
all_metrics.shape
all_metrics['rule_name'].value_counts()

## 9. Model Selection Ranking


In [ ]:
RULE_TO_PRED_CANDIDATES = {
    'cnn_only': ['rule_cnn_only_pred'],
    'distilbert_only': ['rule_distilbert_only_pred'],
    'override': ['rule_override_pred'],
    'weighted_average': ['rule_weighted_average_pred'],
    'cascade_correction': ['rule_cascade_correction_pred'],
    'probabilistic_late': ['rule_probabilistic_late_pred'],
    'custom_uncertain_handoff': ['rule_custom_uncertain_handoff_pred'],
    'cascade_near_boundary_rescue': ['rule_cascade_near_boundary_rescue_pred'],
    'or': ['rule_or_rule_pred', 'rule_or_pred'],
    'or_rule': ['rule_or_rule_pred', 'rule_or_pred'],
    'and': ['rule_and_rule_pred', 'rule_and_pred'],
    'and_rule': ['rule_and_rule_pred', 'rule_and_pred'],
    'ds_fusion': ['rule_ds_fusion_pred'],
    'ds_fusion_no_theta': ['rule_ds_fusion_no_theta_pred'],
}


def get_fusion_pred(pair_df, rule_name):
    rule_name = str(rule_name)
    for col in RULE_TO_PRED_CANDIDATES.get(rule_name, []):
        if col in pair_df.columns:
            return pair_df[col].astype(int)
    if rule_name == 'probabilistic_late' and 'rule_probabilistic_late_prob' in pair_df.columns:
        return (pair_df['rule_probabilistic_late_prob'] >= FUSION_THRESHOLD).astype(int)
    if rule_name == 'ds_fusion_no_theta' and 'rule_ds_fusion_no_theta_prob' in pair_df.columns:
        return (pair_df['rule_ds_fusion_no_theta_prob'] >= FUSION_THRESHOLD).astype(int)
    if rule_name in ['or', 'or_rule']:
        return ((pair_df['cnn_pred'].astype(int) == 1) | (pair_df['dist_pred'].astype(int) == 1)).astype(int)
    if rule_name in ['and', 'and_rule']:
        cnn_pred = pair_df['cnn_pred'].astype(int)
        dist_pred = pair_df['dist_pred'].astype(int)
        return pd.Series(np.where(cnn_pred.eq(dist_pred), cnn_pred, UNKNOWN_PRED), index=pair_df.index).astype(int)
    raise KeyError(f'Could not find or recompute prediction column for rule: {rule_name}')


selection_rows = []

for _, row in all_metrics.iterrows():
    pair_name = row['pair_name']
    rule_name = row['rule_name']
    pair_df = all_pair_matrix_long[all_pair_matrix_long['pair_name'] == pair_name].copy()

    fusion_pred = get_fusion_pred(pair_df, rule_name)
    y_true = pair_df['y_true'].astype(int)
    cnn_pred = pair_df['cnn_pred'].astype(int)

    fusion_known = fusion_pred.isin([0, 1])
    cnn_wrong = cnn_pred.ne(y_true)
    cnn_correct = cnn_pred.eq(y_true)
    danger_samples = y_true.eq(1)
    cnn_wrong_danger = danger_samples & cnn_wrong

    fusion_correct = fusion_pred.eq(y_true) & fusion_known
    fusion_wrong = fusion_pred.ne(y_true) & fusion_known
    fusion_unknown = ~fusion_known

    corrected_cnn_danger_errors = int((cnn_wrong_danger & fusion_correct).sum())
    uncorrected_cnn_danger_errors = int((cnn_wrong_danger & ~fusion_correct).sum())
    unknown_cnn_danger_errors = int((cnn_wrong_danger & fusion_unknown).sum())

    corrected_cnn_errors = int((cnn_wrong & fusion_correct).sum())
    uncorrected_cnn_errors = int((cnn_wrong & ~fusion_correct).sum())
    unknown_cnn_errors = int((cnn_wrong & fusion_unknown).sum())

    cnn_correct_hurt_by_fusion = int((cnn_correct & fusion_wrong).sum())
    cnn_correct_unknown_by_fusion = int((cnn_correct & fusion_unknown).sum())

    net_gain_vs_cnn = corrected_cnn_errors - cnn_correct_hurt_by_fusion
    danger_net_gain_vs_cnn = corrected_cnn_danger_errors - cnn_correct_hurt_by_fusion

    out = row.to_dict()
    out.update({
        'corrected_cnn_danger_errors': corrected_cnn_danger_errors,
        'uncorrected_cnn_danger_errors': uncorrected_cnn_danger_errors,
        'unknown_cnn_danger_errors': unknown_cnn_danger_errors,
        'corrected_cnn_errors': corrected_cnn_errors,
        'uncorrected_cnn_errors': uncorrected_cnn_errors,
        'unknown_cnn_errors': unknown_cnn_errors,
        'cnn_correct_hurt_by_fusion': cnn_correct_hurt_by_fusion,
        'cnn_correct_unknown_by_fusion': cnn_correct_unknown_by_fusion,
        'net_gain_vs_cnn': net_gain_vs_cnn,
        'danger_net_gain_vs_cnn': danger_net_gain_vs_cnn,
    })
    selection_rows.append(out)

model_selection_ranked = pd.DataFrame(selection_rows)
model_selection_ranked = model_selection_ranked.sort_values(
    [
        'corrected_cnn_danger_errors',
        'net_gain_vs_cnn',
        'cnn_correct_hurt_by_fusion',
        'macro_f1',
        'f1_danger',
        'recall_danger',
        'accuracy',
    ],
    ascending=[False, False, True, False, False, False, False],
)

model_selection_ranked.to_csv(
    OUTPUT_DIR / 'model_selection_ranked_by_danger_rescue_then_f1_ALL_RULES.csv',
    index=False,
)

print('Saved ranked model selection to:', OUTPUT_DIR / 'model_selection_ranked_by_danger_rescue_then_f1_ALL_RULES.csv')
print('Rows exported:', len(model_selection_ranked))
print('Rules exported:')
print(model_selection_ranked['rule_name'].value_counts())

display_cols = [
    'cnn_model',
    'distilbert_model',
    'rule_name',
    'corrected_cnn_danger_errors',
    'uncorrected_cnn_danger_errors',
    'unknown_cnn_danger_errors',
    'corrected_cnn_errors',
    'cnn_correct_hurt_by_fusion',
    'net_gain_vs_cnn',
    'danger_net_gain_vs_cnn',
    'macro_f1',
    'f1_danger',
    'recall_danger',
    'precision_danger',
    'accuracy',
    'false_positive_rate',
    'false_negative_rate',
    'unknown_rows',
    'coverage_rate',
]

model_selection_ranked[display_cols].head(30)


## 10. Optional Quick Plots


In [ ]:
plt.figure(figsize=(12, 5))
plot_df = all_metrics.copy()
plot_df.boxplot(column='macro_f1', by='rule_name', rot=45)
plt.title('Macro F1 by Fusion Rule')
plt.suptitle('')
plt.ylabel('Macro F1')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'macro_f1_by_rule_boxplot.png', dpi=200)
plt.show()

plt.figure(figsize=(12, 5))
rule_summary.plot(kind='bar', x='rule_name', y='mean_macro_f1', legend=False, ax=plt.gca())
plt.ylabel('Mean Macro F1')
plt.title('Mean Macro F1 by Rule')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'mean_macro_f1_by_rule.png', dpi=200)
plt.show()


## 11. Inspect One Best Pair


In [ ]:
pair_to_inspect = best_rule_per_pair.iloc[0]['pair_name']
inspect_df = all_pair_matrix_long[all_pair_matrix_long['pair_name'] == pair_to_inspect].copy()

inspect_cols = [
    'filename', 'class', 'y_true', 'reference_label',
    'cnn_prob_danger', 'dist_prob_danger',
    'cnn_pred', 'dist_pred',
    *RULE_PROB_COLS,
    *[col.replace('_prob', '_pred') for col in RULE_PROB_COLS],
    'rule_and_rule_label',
    'cascade_triggered', 'probabilistic_late_triggered', 'custom_uncertain_triggered',
    'near_boundary_rescue_triggered', 'near_boundary_rescue_used_distilbert',
    'or_rule_triggered', 'and_rule_unknown',
]
print(pair_to_inspect)
inspect_df[inspect_cols].head(50)


In [ ]:
# Rank all CNN x DistilBERT x fusion-rule results by danger recall

ranked_by_danger_recall = all_metrics.copy()

# Optional: keep everything, including override.
# Usually exclude only and/and_rule if you do not want unknown-coverage rows.
# Comment this block out if you also want and_rule included.
ranked_by_danger_recall = ranked_by_danger_recall[
    ~ranked_by_danger_recall["rule_name"].isin(["and", "and_rule"])
].copy()

ranked_by_danger_recall = ranked_by_danger_recall.sort_values(
    [
        "recall_danger",
        "f1_danger",
        "macro_f1",
        "precision_danger",
        "accuracy",
    ],
    ascending=[False, False, False, False, False],
).reset_index(drop=True)

display_cols = [
    "cnn_model",
    "distilbert_model",
    "pair_name",
    "rule_name",
    "recall_danger",
    "f1_danger",
    "macro_f1",
    "precision_danger",
    "accuracy",
    "false_positive_rate",
    "false_negative_rate",
    "unknown_rows",
    "coverage_rate",
    "n_rows",
]

display_cols = [col for col in display_cols if col in ranked_by_danger_recall.columns]

ranked_by_danger_recall_export = ranked_by_danger_recall[display_cols]

output_path = OUTPUT_DIR / "all_model_rule_results_ranked_by_danger_recall.csv"
ranked_by_danger_recall_export.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(ranked_by_danger_recall_export))
ranked_by_danger_recall_export.head(50)